# Step 6 — Classification Prompts

**Project:** Prompt Engineering for Clothing Review Analysis  
**Dataset:** Women's Clothing E-Commerce Reviews (`data/reviews.csv`)  
**Goal:** Build and evaluate naive vs. structured sentiment classification prompts against ground-truth ratings and recommendation flags.

In this notebook:
1. We extract a stratified sample of 10 reviews (2 per star rating) with **labels hidden from the model**.
2. We test a naive classification prompt (`classify_v1_naive`) against an improved prompt (`classify_v2_improved`).
3. We compare the model's predictions against the known ground-truth (`Rating` and `Recommended IND`) using an automated answer key.

## 0. Setup and Data Loading

We use a resilient path check to ensure the CSV loads properly regardless of whether Jupyter was launched from the project root or the `prompts/` subfolder.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 120)

# Resilient path check
DATA_PATH = (
    Path("data/reviews.csv")
    if Path("data/reviews.csv").exists()
    else Path("..") / "data" / "reviews.csv"
)

df_raw = pd.read_csv(DATA_PATH, index_col=0)
print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head(3)

Loaded: C:\GamageRecruiters-DataScienceIntern\Month_02\prompt-engineering-task\data\reviews.csv
Raw dataset shape: (23486, 10)


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comfortable,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap...",3,0,0,General,Dresses,Dresses


## 1. Clean Review Text

Reusing findings from our exploratory data analysis:
1. Drop rows with missing `Review Text` (845 empty records).
2. Normalize literal `\r\n` Windows line breaks into single spaces.

In [2]:
df = df_raw.copy()

# 1. Drop records without review text
df = df.dropna(subset=["Review Text"])

# 2. Normalize whitespace and remove literal line breaks
def normalize_review_text(text):
    text = str(text)
    text = text.replace("\r\n", " ")
    text = text.strip()
    return text

df["Review Text"] = df["Review Text"].map(normalize_review_text)
df = df[df["Review Text"].str.len() > 0].copy()

print(f"Cleaned reviews available: {len(df):,}")

Cleaned reviews available: 22,641


## 2. Stratified Classification Sampling

A random sample would be skewed heavily toward 5-star positive reviews (~56% of catalog).

To evaluate classification performance objectively, we draw **2 reviews per star rating** (10 reviews total). 

**Crucial rule:** The prompt given to the LLM must **only** contain the review text and review number. Ground-truth values (`Rating`, `Recommended IND`) must remain hidden so the model cannot cheat.

In [3]:
def pick_classification_sample(df, n=10, seed=7):
    """Sample n reviews evenly across star ratings 1-5 and build prompt text."""
    ratings = [1, 2, 3, 4, 5]
    per_rating = n // len(ratings)
    
    sampled_dfs = []
    for r in ratings:
        group = df[df["Rating"] == r]
        sampled_dfs.append(group.sample(n=per_rating, random_state=seed))
    
    sample_df = pd.concat(sampled_dfs, ignore_index=True)
    # Assign a 1-indexed Review Number
    sample_df["Review Number"] = range(1, len(sample_df) + 1)
    
    # Plain text for the model (ONLY review number and text)
    formatted_lines = ["Here are the customer reviews to classify:\n"]
    for _, row in sample_df.iterrows():
        formatted_lines.append(f"Review {row['Review Number']}:")
        formatted_lines.append(f"{row['Review Text']}\n")
    
    prompt_text = "\n".join(formatted_lines).strip()
    return sample_df, prompt_text

sample_df, formatted_reviews_text = pick_classification_sample(df, n=10, seed=7)

print("Sample distribution across ratings:")
print(sample_df["Rating"].value_counts().sort_index())
print("\nPreview of text sent to the LLM (no ground-truth labels):\n")
print("\n".join(formatted_reviews_text.splitlines()[:6]))

Sample distribution across ratings:
Rating
1    2
2    2
3    2
4    2
5    2
Name: count, dtype: int64

Preview of text sent to the LLM (no ground-truth labels):

Here are the customer reviews to classify:

Review 1:
I tried this shirt at my local retailer and absolutely loved it ...until i read dry clean only!!! really retailer? i know you can do better than that...

Review 2:


## 3. Prompt Engineering: Classification

- `classify_v1_naive`: An open-ended command with no label constraints or formatting structure.
- `classify_v2_improved`: Assigns a domain-specific persona, restricts labels strictly to a closed set (`Positive`, `Negative`, `Neutral`), defines classification guidelines, and mandates a scannable Markdown table output.

In [4]:
# Version 1: Naive
classify_v1_naive = "Tell me if these reviews are good or bad."

# Version 2: Improved
classify_v2_improved = """
You are an e-commerce customer sentiment classification specialist.

Task:
Classify EACH customer review below into exactly ONE of the following three sentiment labels:
- Positive
- Negative
- Neutral

Classification Criteria:
- Positive: Clear satisfaction, praise, or enthusiasm.
- Negative: Clear dissatisfaction, defective product, severe fit flaws, or return statement.
- Neutral: Mixed feedback (balanced pros and cons) or indifferent tone.

Output Requirements:
Return your response strictly as a Markdown table with the following three columns:
| Review Number | Label | One-word reason |

Rules:
1. Do not use any label other than Positive, Negative, or Neutral.
2. Include every review from 1 to 10 in sequential order.
3. Do not include any introductory or concluding text.
""".strip()

## 4. Copy-Ready Prompts

Run these cells, copy the blocks, and paste them into Claude, ChatGPT, or your preferred LLM interface.

In [5]:
print("=" * 72)
print("COPY FROM HERE — classify_v1 (naive)")
print("=" * 72)
print(f"{classify_v1_naive}\n\n{formatted_reviews_text}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — classify_v1 (naive)
Tell me if these reviews are good or bad.

Here are the customer reviews to classify:

Review 1:
I tried this shirt at my local retailer and absolutely loved it ...until i read dry clean only!!! really retailer? i know you can do better than that...

Review 2:
Loved how this draped and im super petite 5'1"(32b bust) and this was seriously one of the most ill fitting items i have every owned. i purchased the pxs and it was huge. i hope this has a better fit as the other reviewers seem to have a more positive review. i was just appalled as to how bad the measurements were off. unfortunately the straps are not adjustable so this went below my chest. too sad retailer, please get the fit right!!

Review 3:
I wanted to love this sweater and it has some nice distinctive features that make it more interesting than your average cardigan. unfortunately, however, i found that flared in an unflattering manner over my stomach. the chest fit well - though it was 

In [6]:
print("=" * 72)
print("COPY FROM HERE — classify_v2 (improved)")
print("=" * 72)
print(f"{classify_v2_improved}\n\n{formatted_reviews_text}")
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)

COPY FROM HERE — classify_v2 (improved)
You are an e-commerce customer sentiment classification specialist.

Task:
Classify EACH customer review below into exactly ONE of the following three sentiment labels:
- Positive
- Negative
- Neutral

Classification Criteria:
- Positive: Clear satisfaction, praise, or enthusiasm.
- Negative: Clear dissatisfaction, defective product, severe fit flaws, or return statement.
- Neutral: Mixed feedback (balanced pros and cons) or indifferent tone.

Output Requirements:
Return your response strictly as a Markdown table with the following three columns:
| Review Number | Label | One-word reason |

Rules:
1. Do not use any label other than Positive, Negative, or Neutral.
2. Include every review from 1 to 10 in sequential order.
3. Do not include any introductory or concluding text.

Here are the customer reviews to classify:

Review 1:
I tried this shirt at my local retailer and absolutely loved it ...until i read dry clean only!!! really retailer? i kno

## 5. Ground-Truth Answer Key

**Check this cell AFTER running the prompts.**

This table maps each review back to its true `Rating` and `Recommended IND` flag to score the model's accuracy.

In [8]:
answer_key = sample_df[["Review Number", "Rating", "Recommended IND", "Class Name", "Review Text"]].copy()

# Derive expected baseline sentiment based on ratings
def expected_sentiment(row):
    if row["Rating"] in [1, 2]:
        return "Negative"
    elif row["Rating"] == 3:
        return "Neutral / Mixed"
    else:
        return "Positive"

answer_key["Expected Baseline"] = answer_key.apply(expected_sentiment, axis=1)

print("=== GROUND-TRUTH ANSWER KEY ===")
display(answer_key[["Review Number", "Rating", "Recommended IND", "Expected Baseline", "Class Name"]])

print("\nReview Snippets for Sanity Checking:")
for _, row in answer_key.iterrows():
    rec_str = "Recommended" if row["Recommended IND"] == 1 else "Not Recommended"
    print(f"Review {row['Review Number']} [{row['Rating']}★ | {rec_str}]: {row['Review Text'][:100]}...")

=== GROUND-TRUTH ANSWER KEY ===


,Review Number,Rating,Recommended IND,Expected Baseline,Class Name
0,1,1,0,Negative,Blouses
1,2,1,0,Negative,Intimates
2,3,2,0,Negative,Fine gauge
3,4,2,0,Negative,Dresses
4,5,3,0,Neutral / Mixed,Blouses
5,6,3,1,Neutral / Mixed,Dresses
6,7,4,1,Positive,Blouses
7,8,4,1,Positive,Dresses
8,9,5,1,Positive,Dresses
9,10,5,1,Positive,Knits



Review Snippets for Sanity Checking:
Review 1 [1★ | Not Recommended]: I tried this shirt at my local retailer and absolutely loved it ...until i read dry clean only!!! re...
Review 2 [1★ | Not Recommended]: Loved how this draped and im super petite 5'1"(32b bust) and this was seriously one of the most ill ...
Review 3 [2★ | Not Recommended]: I wanted to love this sweater and it has some nice distinctive features that make it more interestin...
Review 4 [2★ | Not Recommended]: Good reason to read reviews. i concur with others in that it fins large and looks boxy. heavy fabric...
Review 5 [3★ | Not Recommended]: Great idea, but this was not a flattering top on me. it almost worked, but ended up making me look l...
Review 6 [3★ | Recommended]: The fabric has a beautiful drape; the skirt is a nice length, and it blows, moves, and swings in a m...
Review 7 [4★ | Recommended]: Love the colors! love the stripes! great shape! great style! but...i am normally a 12 on top...and t...
Review 8 [4

## Output — v1 (paste here)

* **Review 1: Bad** (Negative) — Despite liking the initial look, the "dry clean only" requirement was an immediate dealbreaker and generated frustration.
* **Review 2: Bad** (Negative) — Extreme sizing issues; the reviewer found it "appalled" by the fit, "huge," and unusable because non-adjustable straps sat below the chest.
* **Review 3: Bad** (Negative) — While they liked the style conceptually, the unflattering fit ("three sizes larger"), thin fabric, and durability concerns make it an overall negative assessment.
* **Review 4: Bad** (Negative) — Direct complaint regarding oversized proportions, boxy shape, and excessively heavy fabric.
* **Review 5: Bad** (Negative) — Unflattering silhouette ("makes me look like i am pregnant") and poor value for the price, leading to rejection.
* **Review 6: Mixed / Leaning Bad** (Neutral to Negative for the buyer) — Appreciates the aesthetic design ("stunning top") and styling versatility, but concludes it fails in fit and looks like maternity wear on their body.

## Output — v2 (paste here)


| Review Number | Label | One-word reason |
| :---: | :---: | :---: |
| 1 | Negative | Care |
| 2 | Negative | Fit |
| 3 | Negative | Unflattering |
| 4 | Negative | Fit |
| 5 | Negative | Flattering |
| 6 | Neutral | Mixed |
| 7 | Neutral | Fit |
| 8 | Neutral | Mixed |
| 9 | Positive | Compliments |
| 10 | Negative | Fit |

## Comparison notes

- **Accuracy of v2 vs. Ground Truth:**
  - **Baseline Exact Match:** 6/10 (60%) agreement with the star rating heuristic (Rating 1-2 = Negative, 3 = Neutral, 4-5 = Positive).
  - **High-Dissatisfaction Precision:** 100% (4/4) on true 1-star and 2-star negative reviews (Reviews 1–4).
  - **Edge Cases & Discrepancies:**
    - **Review 5 (3★, Not Recommended):** Classified as Negative instead of Neutral because the customer explicitly stated the top "did not justify a place in my closet." The model accurately detected negative intent over the neutral star score.
    - **Reviews 7 & 8 (4★, Recommended):** Classified as Neutral because shoppers discussed sizing exchanges and polarized feedback, muting overall positive sentiment.
    - **Review 10 (5★, Recommended):** Marked as Negative due to fit complaints ("looked like a maternity top on me"). This highlights how zero-shot sentiment can decouple from numerical ratings when customer review text emphasizes personal fit failure despite overall product admiration.

- **Structural Shortcomings of v1:**
  - Lacked a closed label taxonomy, outputting inconsistent conversational classifications ("Bad", "Mixed / Leaning Bad") alongside unstructured prose.
  - Omitted half the evaluation set (dropped Reviews 7–10), making automated parsing and mathematical scoring impossible.

- **Why v2 Enabled Quantitative Measurement:**
  - Enforced a strict 3-class schema (`Positive`, `Negative`, `Neutral`) and a 3-column Markdown table.
  - Allowed direct, row-by-row scoring against ground-truth dataset labels (`Rating` and `Recommended IND`) without programmatic post-processing.